# Notebook 00: Inference Basics - How Autoregressive Generation Works

## 📚 Historical Context

**Timeline**: 2017-2019 - Foundation of autoregressive generation

**The Problem**:
- Transformers (2017) introduced parallel training, but inference is sequential
- GPT-1 (2018): 117M params, naive inference = 5-10 tokens/sec
- GPT-2 (2019): 1.5B params, performance became a serious bottleneck
- No standardized understanding of inference mechanics

**The Reality**:
- Training is parallel (process entire sequence at once)
- Inference is sequential (generate one token at a time)
- This fundamental difference shapes all optimization techniques

## 🎯 What You'll Learn

1. How autoregressive generation works step-by-step
2. Forward pass mechanics during inference
3. Token-by-token generation loop
4. Sampling strategies (greedy, top-k, top-p, temperature)
5. Working code examples with GPT-2
6. Visualizations of the generation process

## 💡 Why This Matters

- **Foundation**: Every optimization builds on understanding this
- **Bottleneck identification**: Know what's slow before fixing it
- **Sampling strategies**: Different methods for different use cases
- **Real-world impact**: Chat applications, code generation, creative writing

**Reference**: See LLM_INFERENCE_ROADMAP.md - Stage 0: Inference Fundamentals

---

## Setup and Imports

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("=" * 80)
print("ENVIRONMENT CHECK")
print("=" * 80)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("=" * 80)

## Part 1: The Autoregressive Generation Process

### What is Autoregressive Generation?

**Definition**: Generate sequence one token at a time, where each new token depends on all previous tokens.

### The Process:
```
Step 1: Input: "The cat"           → Model → Predict: "sat"
Step 2: Input: "The cat sat"       → Model → Predict: "on"
Step 3: Input: "The cat sat on"    → Model → Predict: "the"
Step 4: Input: "The cat sat on the" → Model → Predict: "mat"
```

### Why Sequential?
- Cannot predict token N+2 without knowing token N+1
- Creates fundamental latency constraint
- This is THE bottleneck that all inference optimization tries to work around

### Training vs Inference:

**Training** (Parallel):
```python
# Input: "The cat sat on the mat"
# Process ALL tokens simultaneously
logits = model(input_ids)  # One forward pass for entire sequence
```

**Inference** (Sequential):
```python
# Generate token by token
for i in range(max_new_tokens):
    logits = model(input_ids)  # Forward pass for each new token
    next_token = sample(logits)
    input_ids = append(input_ids, next_token)
```

## Part 2: Load Model for Demonstration

We'll use **GPT-2 Small** (124M parameters) for demonstrations:
- Small enough to load quickly
- Large enough to show real behavior
- Same architecture as modern LLMs (just smaller)

In [ ]:
# Load GPT-2 model and tokenizer
model_name = "gpt2"  # 124M parameters

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()  # Set to evaluation mode (disables dropout)

# Model statistics
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: {model_name}")
print(f"Total parameters: {total_params:,}")
print(f"Device: {device}")
print(f"Vocabulary size: {len(tokenizer)}")

# Check model architecture
print(f"\nArchitecture:")
print(f"  Hidden size: {model.config.hidden_size}")
print(f"  Number of layers: {model.config.num_hidden_layers}")
print(f"  Number of attention heads: {model.config.num_attention_heads}")
print(f"  Max position embeddings: {model.config.max_position_embeddings}")

## Part 3: Naive Inference - Step by Step

Let's implement the most basic inference loop to understand what's happening.

### The Algorithm:
```
1. Start with input prompt
2. Run forward pass through entire model
3. Get logits for next token
4. Sample next token from logits
5. Append token to input
6. Repeat until stopping condition
```

### Key Insight:
The model processes the ENTIRE sequence every time, even though we only need predictions for the last token. This is wasteful and will be optimized in Stage 1 with KV caching.

In [ ]:
def naive_generate(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 20,
    temperature: float = 1.0,
    verbose: bool = True
) -> Tuple[str, List[float]]:
    """
    Naive autoregressive generation - no optimizations.
    
    Args:
        model: The language model
        tokenizer: The tokenizer
        prompt: Input text to start generation
        max_new_tokens: Maximum number of tokens to generate
        temperature: Sampling temperature (higher = more random)
        verbose: Print step-by-step information
    
    Returns:
        generated_text: The complete generated text
        token_times: Time taken for each token generation
    """
    # Tokenize input prompt
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    if verbose:
        print("=" * 80)
        print("NAIVE GENERATION - STEP BY STEP")
        print("=" * 80)
        print(f"Prompt: '{prompt}'")
        print(f"Prompt tokens: {input_ids.shape[1]}")
        print("=" * 80)
    
    token_times = []
    generated_tokens = []
    
    # Generation loop
    for step in range(max_new_tokens):
        step_start = time.time()
        
        # Forward pass through entire model
        # NOTE: This processes ALL tokens every time, even the ones we already processed
        # This is the main inefficiency of naive inference
        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits  # Shape: [batch_size, sequence_length, vocab_size]
        
        # Get logits for the last token (this is what we predict next)
        next_token_logits = logits[:, -1, :]  # Shape: [batch_size, vocab_size]
        
        # Apply temperature
        # Temperature controls randomness:
        # - temperature < 1.0: More deterministic (sharper distribution)
        # - temperature = 1.0: Normal distribution
        # - temperature > 1.0: More random (flatter distribution)
        if temperature != 1.0:
            next_token_logits = next_token_logits / temperature
        
        # Convert logits to probabilities
        probs = F.softmax(next_token_logits, dim=-1)
        
        # Sample next token
        next_token = torch.multinomial(probs, num_samples=1)  # Shape: [batch_size, 1]
        
        # Append to input sequence
        input_ids = torch.cat([input_ids, next_token], dim=-1)
        
        step_time = time.time() - step_start
        token_times.append(step_time)
        
        # Decode for display
        token_text = tokenizer.decode(next_token[0])
        generated_tokens.append(token_text)
        
        if verbose:
            # Get top-5 tokens for this position
            top5_probs, top5_indices = torch.topk(probs[0], k=5)
            top5_tokens = [tokenizer.decode([idx]) for idx in top5_indices]
            
            print(f"\nStep {step + 1}:")
            print(f"  Input length: {input_ids.shape[1]} tokens")
            print(f"  Selected token: '{token_text}' (id: {next_token.item()})")
            print(f"  Top-5 alternatives: {list(zip(top5_tokens, top5_probs.cpu().numpy()))}")
            print(f"  Time: {step_time*1000:.2f} ms")
        
        # Check for end of sequence token
        if next_token.item() == tokenizer.eos_token_id:
            if verbose:
                print("\n  → End of sequence token reached")
            break
    
    # Decode final output
    generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    
    if verbose:
        print("\n" + "=" * 80)
        print("GENERATION COMPLETE")
        print("=" * 80)
        print(f"Total tokens generated: {len(generated_tokens)}")
        print(f"Average time per token: {np.mean(token_times)*1000:.2f} ms")
        print(f"Total time: {sum(token_times):.3f} s")
        print(f"\nGenerated text:\n{generated_text}")
        print("=" * 80)
    
    return generated_text, token_times

# Test naive generation
prompt = "The future of artificial intelligence is"
text, times = naive_generate(model, tokenizer, prompt, max_new_tokens=15, temperature=0.8)

## Part 4: Understanding the Forward Pass

### What Happens During Forward Pass:

```
Input tokens → Embedding → Transformer Layers → Language Model Head → Logits
```

### Detailed Breakdown:

1. **Token Embedding** (lookup): Convert token IDs to vectors
2. **Position Embedding** (lookup): Add positional information
3. **Transformer Layers** (compute-heavy):
   - Self-attention: Look at all previous tokens
   - Feed-forward network: Process each position
   - Repeat for N layers
4. **Language Model Head** (linear layer): Project to vocabulary size
5. **Logits**: Raw scores for each token in vocabulary

### Memory Access Pattern:
- **Bottleneck**: Loading model weights from GPU memory
- GPT-2 (124M params) = ~248MB in FP16
- Must load ALL weights for each token generation
- This is why inference is **memory-bandwidth bound**

In [ ]:
def analyze_forward_pass(model, tokenizer, prompt: str):
    """
    Analyze what happens during a single forward pass.
    """
    print("=" * 80)
    print("FORWARD PASS ANALYSIS")
    print("=" * 80)
    
    # Tokenize
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    print(f"\n1. INPUT:")
    print(f"   Prompt: '{prompt}'")
    print(f"   Token IDs: {input_ids[0].tolist()}")
    print(f"   Tokens: {[tokenizer.decode([t]) for t in input_ids[0]]}")
    print(f"   Shape: {input_ids.shape}")
    
    # Forward pass with output inspection
    with torch.no_grad():
        # Get embeddings
        inputs_embeds = model.transformer.wte(input_ids)  # Word token embeddings
        position_ids = torch.arange(0, input_ids.shape[1], dtype=torch.long, device=device)
        position_embeds = model.transformer.wpe(position_ids)  # Position embeddings
        
        print(f"\n2. EMBEDDINGS:")
        print(f"   Token embeddings shape: {inputs_embeds.shape}")
        print(f"   Position embeddings shape: {position_embeds.shape}")
        print(f"   Hidden size: {model.config.hidden_size}")
        
        # Full forward pass
        start = time.time()
        outputs = model(input_ids, output_hidden_states=True)
        forward_time = time.time() - start
        
        logits = outputs.logits
        hidden_states = outputs.hidden_states
        
        print(f"\n3. TRANSFORMER LAYERS:")
        print(f"   Number of layers: {len(hidden_states) - 1}")
        print(f"   Hidden state shape per layer: {hidden_states[0].shape}")
        print(f"   Forward pass time: {forward_time*1000:.2f} ms")
        
        print(f"\n4. OUTPUT:")
        print(f"   Logits shape: {logits.shape}")
        print(f"   Vocabulary size: {logits.shape[-1]}")
        
        # Analyze last token prediction
        last_token_logits = logits[0, -1, :]
        probs = F.softmax(last_token_logits, dim=-1)
        
        # Top-k predictions
        top_k = 10
        top_probs, top_indices = torch.topk(probs, k=top_k)
        
        print(f"\n5. TOP-{top_k} PREDICTIONS FOR NEXT TOKEN:")
        for i, (prob, idx) in enumerate(zip(top_probs, top_indices)):
            token = tokenizer.decode([idx])
            print(f"   {i+1}. '{token}' - probability: {prob.item():.4f}")
        
        print("\n" + "=" * 80)
        print("KEY OBSERVATIONS:")
        print("=" * 80)
        print("• Model processes ENTIRE sequence to predict ONE token")
        print("• All model weights loaded from memory for each forward pass")
        print("• This is the fundamental inefficiency we'll optimize in Stage 1")
        print("=" * 80)

# Analyze a forward pass
analyze_forward_pass(model, tokenizer, "The cat sat")

## Part 5: Sampling Strategies

Different ways to select the next token from the probability distribution.

### Strategy Comparison:

| Strategy | Description | Use Case | Quality | Diversity |
|----------|-------------|----------|---------|----------|
| **Greedy** | Always pick highest probability | Factual Q&A, translation | High | Low |
| **Temperature** | Control randomness | Adjust creativity | Varies | Varies |
| **Top-K** | Sample from top K tokens | Balanced generation | Medium | Medium |
| **Top-P (Nucleus)** | Sample from cumulative prob P | Creative writing | Medium | High |
| **Beam Search** | Keep top N sequences | Translation, summarization | Highest | Lowest |

### Temperature Effect:
```
Temperature = 0.1:  [0.95, 0.03, 0.02] → Very deterministic
Temperature = 1.0:  [0.60, 0.25, 0.15] → Balanced
Temperature = 2.0:  [0.40, 0.35, 0.25] → Very random
```

In [ ]:
def generate_with_strategy(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 30,
    strategy: str = "greedy",
    temperature: float = 1.0,
    top_k: int = 50,
    top_p: float = 0.95,
) -> str:
    """
    Generate text using different sampling strategies.
    
    Args:
        strategy: One of ['greedy', 'temperature', 'top_k', 'top_p']
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits[:, -1, :]
        
        # Apply different strategies
        if strategy == "greedy":
            # Always pick the highest probability token
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
        
        elif strategy == "temperature":
            # Apply temperature and sample
            logits = logits / temperature
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        
        elif strategy == "top_k":
            # Keep only top-k tokens
            top_k_logits, top_k_indices = torch.topk(logits, top_k)
            # Set all other logits to -inf
            logits_filtered = torch.full_like(logits, float('-inf'))
            logits_filtered.scatter_(1, top_k_indices, top_k_logits)
            # Sample from filtered distribution
            probs = F.softmax(logits_filtered / temperature, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        
        elif strategy == "top_p":
            # Nucleus sampling: keep tokens with cumulative probability >= top_p
            probs = F.softmax(logits / temperature, dim=-1)
            sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
            
            # Remove tokens with cumulative probability above threshold
            sorted_indices_to_remove = cumulative_probs > top_p
            # Keep at least one token
            sorted_indices_to_remove[..., 0] = False
            
            # Create filtered distribution
            indices_to_remove = sorted_indices_to_remove.scatter(
                1, sorted_indices, sorted_indices_to_remove
            )
            logits_filtered = logits.masked_fill(indices_to_remove, float('-inf'))
            probs = F.softmax(logits_filtered / temperature, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        
        # Append to sequence
        input_ids = torch.cat([input_ids, next_token], dim=-1)
        
        # Stop if EOS
        if next_token.item() == tokenizer.eos_token_id:
            break
    
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

# Compare different sampling strategies
print("=" * 80)
print("COMPARING SAMPLING STRATEGIES")
print("=" * 80)

prompt = "Once upon a time in a land far away"
print(f"Prompt: '{prompt}'\n")

strategies = [
    ("greedy", {}),
    ("temperature", {"temperature": 0.7}),
    ("top_k", {"top_k": 50, "temperature": 0.8}),
    ("top_p", {"top_p": 0.9, "temperature": 0.8}),
]

for strategy, params in strategies:
    print(f"\n{'='*40}")
    print(f"Strategy: {strategy.upper()}")
    if params:
        print(f"Parameters: {params}")
    print(f"{'='*40}")
    
    result = generate_with_strategy(
        model, tokenizer, prompt,
        max_new_tokens=25,
        strategy=strategy,
        **params
    )
    print(result)

print("\n" + "=" * 80)

## Part 6: Visualizing the Generation Process

In [ ]:
def visualize_generation_timeline(token_times: List[float]):
    """
    Visualize time taken for each token generation.
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Time per token
    ax = axes[0, 0]
    ax.plot(range(1, len(token_times) + 1), [t*1000 for t in token_times], 
            marker='o', linewidth=2, markersize=6)
    ax.set_xlabel('Token Number', fontsize=12)
    ax.set_ylabel('Time (ms)', fontsize=12)
    ax.set_title('Time per Token Generation', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # 2. Cumulative time
    ax = axes[0, 1]
    cumulative_times = np.cumsum(token_times)
    ax.plot(range(1, len(cumulative_times) + 1), cumulative_times, 
            marker='o', color='green', linewidth=2, markersize=6)
    ax.set_xlabel('Token Number', fontsize=12)
    ax.set_ylabel('Cumulative Time (s)', fontsize=12)
    ax.set_title('Cumulative Generation Time', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # 3. Time distribution histogram
    ax = axes[1, 0]
    ax.hist([t*1000 for t in token_times], bins=15, color='skyblue', edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(token_times)*1000, color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(token_times)*1000:.2f} ms')
    ax.set_xlabel('Time (ms)', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title('Distribution of Token Generation Times', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # 4. Statistics
    ax = axes[1, 1]
    ax.axis('off')
    stats_text = f"""
    GENERATION STATISTICS
    
    Total tokens: {len(token_times)}
    Total time: {sum(token_times):.3f} s
    
    Mean time/token: {np.mean(token_times)*1000:.2f} ms
    Median time/token: {np.median(token_times)*1000:.2f} ms
    Std dev: {np.std(token_times)*1000:.2f} ms
    
    Min time: {min(token_times)*1000:.2f} ms
    Max time: {max(token_times)*1000:.2f} ms
    
    Tokens per second: {len(token_times)/sum(token_times):.2f}
    """
    ax.text(0.1, 0.5, stats_text, fontsize=12, family='monospace',
            verticalalignment='center')
    
    plt.tight_layout()
    plt.show()

# Generate and visualize
prompt = "The advancement of artificial intelligence has led to"
print(f"Generating text from prompt: '{prompt}'\n")
text, times = naive_generate(model, tokenizer, prompt, max_new_tokens=30, temperature=0.8, verbose=False)
print(f"\nGenerated: {text}\n")
visualize_generation_timeline(times)

## Part 7: Probability Distribution Visualization

Let's visualize how different sampling strategies affect token selection.

In [ ]:
def visualize_sampling_strategies(model, tokenizer, prompt: str):
    """
    Visualize how different sampling strategies modify probability distributions.
    """
    # Get logits for next token
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits[0, -1, :]
    
    # Get top 30 tokens for visualization
    top_k = 30
    original_probs = F.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(original_probs, k=top_k)
    top_tokens = [tokenizer.decode([idx]) for idx in top_indices]
    
    # Create different distributions
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Original distribution
    ax = axes[0, 0]
    ax.bar(range(top_k), top_probs.cpu().numpy(), color='steelblue', alpha=0.7)
    ax.set_xlabel('Token Rank', fontsize=11)
    ax.set_ylabel('Probability', fontsize=11)
    ax.set_title('Original Distribution (Temperature=1.0)', fontsize=13, fontweight='bold')
    ax.set_xticks(range(0, top_k, 3))
    
    # Add top-3 token labels
    for i in range(3):
        ax.text(i, top_probs[i].item() + 0.01, top_tokens[i], 
                ha='center', fontsize=9, fontweight='bold')
    
    # 2. Low temperature (more deterministic)
    ax = axes[0, 1]
    temp_low = 0.3
    low_temp_logits = logits / temp_low
    low_temp_probs = F.softmax(low_temp_logits, dim=-1)
    low_temp_top_probs = low_temp_probs[top_indices]
    ax.bar(range(top_k), low_temp_top_probs.cpu().numpy(), color='red', alpha=0.7)
    ax.set_xlabel('Token Rank', fontsize=11)
    ax.set_ylabel('Probability', fontsize=11)
    ax.set_title(f'Low Temperature={temp_low} (More Deterministic)', fontsize=13, fontweight='bold')
    ax.set_xticks(range(0, top_k, 3))
    
    for i in range(3):
        ax.text(i, low_temp_top_probs[i].item() + 0.01, top_tokens[i], 
                ha='center', fontsize=9, fontweight='bold')
    
    # 3. High temperature (more random)
    ax = axes[1, 0]
    temp_high = 2.0
    high_temp_logits = logits / temp_high
    high_temp_probs = F.softmax(high_temp_logits, dim=-1)
    high_temp_top_probs = high_temp_probs[top_indices]
    ax.bar(range(top_k), high_temp_top_probs.cpu().numpy(), color='orange', alpha=0.7)
    ax.set_xlabel('Token Rank', fontsize=11)
    ax.set_ylabel('Probability', fontsize=11)
    ax.set_title(f'High Temperature={temp_high} (More Random)', fontsize=13, fontweight='bold')
    ax.set_xticks(range(0, top_k, 3))
    
    for i in range(3):
        ax.text(i, high_temp_top_probs[i].item() + 0.01, top_tokens[i], 
                ha='center', fontsize=9, fontweight='bold')
    
    # 4. Top-P (nucleus) sampling visualization
    ax = axes[1, 1]
    sorted_probs, _ = torch.sort(original_probs, descending=True)
    cumulative = torch.cumsum(sorted_probs[:top_k], dim=0)
    ax.bar(range(top_k), sorted_probs[:top_k].cpu().numpy(), 
           color='green', alpha=0.6, label='Probability')
    ax.plot(range(top_k), cumulative.cpu().numpy(), 
            color='red', linewidth=3, marker='o', markersize=4, label='Cumulative')
    ax.axhline(y=0.9, color='purple', linestyle='--', linewidth=2, label='Top-P=0.9 threshold')
    ax.set_xlabel('Token Rank', fontsize=11)
    ax.set_ylabel('Probability', fontsize=11)
    ax.set_title('Top-P (Nucleus) Sampling', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_xticks(range(0, top_k, 3))
    
    plt.suptitle(f'Prompt: "{prompt}"', fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()
    
    # Print top tokens with probabilities
    print("\n" + "=" * 80)
    print("TOP 10 TOKENS WITH PROBABILITIES")
    print("=" * 80)
    for i in range(min(10, top_k)):
        token = top_tokens[i]
        prob = top_probs[i].item()
        print(f"{i+1:2d}. '{token:15s}' → {prob:.4f} ({prob*100:.2f}%)")
    print("=" * 80)

# Visualize sampling
prompt = "The most important factor in machine learning is"
visualize_sampling_strategies(model, tokenizer, prompt)

## Part 8: Performance Baseline

Let's establish a performance baseline for naive inference.
This will be our comparison point for all future optimizations.

In [ ]:
def benchmark_naive_inference(
    model, 
    tokenizer, 
    num_runs: int = 5,
    max_new_tokens: int = 50
):
    """
    Benchmark naive inference performance.
    """
    print("=" * 80)
    print("NAIVE INFERENCE BENCHMARK")
    print("=" * 80)
    print(f"Runs: {num_runs}")
    print(f"Max new tokens: {max_new_tokens}")
    print(f"Device: {device}")
    print("=" * 80)
    
    prompts = [
        "The future of technology is",
        "In the year 2030, we will see",
        "The most important invention of the 21st century is",
        "Artificial intelligence will transform",
        "The key to solving climate change is",
    ]
    
    all_results = []
    
    for run_idx in range(num_runs):
        prompt = prompts[run_idx % len(prompts)]
        print(f"\nRun {run_idx + 1}/{num_runs}: '{prompt[:40]}...'")
        
        _, token_times = naive_generate(
            model, tokenizer, prompt,
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            verbose=False
        )
        
        results = {
            'total_time': sum(token_times),
            'num_tokens': len(token_times),
            'tokens_per_sec': len(token_times) / sum(token_times),
            'mean_latency': np.mean(token_times) * 1000,  # ms
            'median_latency': np.median(token_times) * 1000,
        }
        all_results.append(results)
        
        print(f"  Tokens/sec: {results['tokens_per_sec']:.2f}")
        print(f"  Mean latency: {results['mean_latency']:.2f} ms/token")
    
    # Aggregate statistics
    print("\n" + "=" * 80)
    print("AGGREGATE RESULTS")
    print("=" * 80)
    
    avg_tokens_per_sec = np.mean([r['tokens_per_sec'] for r in all_results])
    avg_latency = np.mean([r['mean_latency'] for r in all_results])
    std_tokens_per_sec = np.std([r['tokens_per_sec'] for r in all_results])
    std_latency = np.std([r['mean_latency'] for r in all_results])
    
    print(f"Average throughput: {avg_tokens_per_sec:.2f} ± {std_tokens_per_sec:.2f} tokens/sec")
    print(f"Average latency: {avg_latency:.2f} ± {std_latency:.2f} ms/token")
    print("\n" + "=" * 80)
    print("BASELINE ESTABLISHED")
    print("=" * 80)
    print("This is the performance of NAIVE inference with NO optimizations.")
    print("Expected improvements in future stages:")
    print("  • Stage 1 (KV Cache + FP16): 5-10x speedup")
    print("  • Stage 2 (Flash Attention): Additional 1.5-2x speedup")
    print("  • Stage 3 (vLLM): 3-10x throughput improvement")
    print("  • Stage 4 (Quantization + Speculative): 2-4x additional speedup")
    print("=" * 80)
    
    return all_results

# Run benchmark
results = benchmark_naive_inference(model, tokenizer, num_runs=5, max_new_tokens=30)

## 📊 Summary and Key Takeaways

### What We Learned:

1. **Autoregressive Generation**:
   - Generate one token at a time
   - Each token depends on all previous tokens
   - Fundamentally sequential process

2. **Forward Pass Mechanics**:
   - Process ENTIRE sequence for each new token
   - Model weights loaded from memory every time
   - Memory-bandwidth is the primary bottleneck

3. **Sampling Strategies**:
   - Greedy: Deterministic, best for factual tasks
   - Temperature: Control randomness
   - Top-K: Limit to top K probable tokens
   - Top-P: Dynamic cutoff based on cumulative probability

4. **Performance Characteristics**:
   - GPT-2 (124M): ~5-15 tokens/sec on GPU (naive)
   - Larger models are slower (more memory to load)
   - Latency increases linearly with sequence length

### The Inefficiency:

```
Token 1: Process [A]           → Generate B
Token 2: Process [A, B]        → Generate C  (recomputes A!)
Token 3: Process [A, B, C]     → Generate D  (recomputes A, B!)
Token 4: Process [A, B, C, D]  → Generate E  (recomputes A, B, C!)
```

**The Solution**: KV Caching (Stage 1) - cache intermediate results to avoid recomputation.

### Key Metrics:

- **Throughput**: Tokens generated per second
- **Latency**: Time per token generation
- **Time to First Token (TTFT)**: Latency for first generated token
- **Memory Usage**: GPU memory consumed

### Next Steps:

**Notebook 01: Profiling Inference** - Learn to:
- Use PyTorch Profiler
- Identify bottlenecks (memory vs compute)
- Analyze GPU utilization
- Measure memory bandwidth

---

### Historical Context:

**2017-2019**: The "naive era" of inference
- No KV caching (recomputed everything)
- No batching (one request at a time)
- FP32 precision (2x memory of FP16)
- Performance: 5-10 tokens/sec for GPT-2

**Today**: With all optimizations
- KV caching (Stage 1): 5-10x faster
- Batching (Stage 1): 5-8x throughput
- Flash Attention (Stage 2): 1.5-2x faster
- vLLM (Stage 3): 3-10x throughput
- Quantization (Stage 4): 2-4x faster
- **Total improvement**: 100-500x better than naive!

---

**Reference**: LLM_INFERENCE_ROADMAP.md - Stage 0